In [1]:
import os
from pathlib import Path

os.environ["HF_HOME"] = "/scratch/legal/cache"
os.environ["HF_HUB_CACHE"] = "/scratch/legal/cache/hub"
os.environ["TRANSFORMERS_CACHE"] = "/scratch/legal/cache/transformers"
os.environ["XDG_CACHE_HOME"] = "/scratch/legal/cache"

import torch
from transformers import (
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
    AutoTokenizer,
)

/home/data/legal/smulever/venv/lib64/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cargamos llama (sin cuantizar)

In [2]:
RUTA_MODELO = Path("/scratch/legal/modelos/instruct/Llama-3.1-8B-Instruct")

tok = AutoTokenizer.from_pretrained(RUTA_MODELO)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(
    RUTA_MODELO,
    torch_dtype=torch.bfloat16,
    device_map={"": 0},
)
model.eval()

print("llama:", next(model.parameters()).device)
torch.cuda.synchronize()
print("sync OK después de Llama")
print(f"alloc MB: {torch.cuda.memory_allocated(0)/1024**2:.1f}")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 291/291 [00:01<00:00, 152.16it/s]


llama: cuda:0
sync OK después de Llama
alloc MB: 15316.5


## ¿Qué es un NLI?

**NLI** = *Natural Language Inference* (inferencia en lenguaje natural).

Es un modelo que, dada una **premisa** y una **hipótesis**, decide cómo se relacionan lógicamente. No genera texto: **clasifica** la relación.

En este experimento lo usamos para saber si dos respuestas del LLM dicen **lo mismo** (equivalencia semántica): chequeamos ida y vuelta (A→B y B→A).

### Las 3 salidas posibles

1. **Entailment (implicación)**  
   La hipótesis **se deduce** de la premisa. Si la premisa es verdadera, la hipótesis también debe serlo.  
   Ejemplo: premisa “el inquilino no pagó el alquiler” → hipótesis “hubo falta de pago”.

2. **Neutral**  
   La premisa **no alcanza** para afirmar ni negar la hipótesis. Puede ser verdad o no.  
   Ejemplo: premisa “el inquilino no pagó” → hipótesis “el dueño no quiso negociar”.

3. **Contradiction (contradicción)**  
   La hipótesis **choca** con la premisa; no pueden ser verdaderas a la vez.  
   Ejemplo: premisa “el inquilino pagó toda la deuda” → hipótesis “el inquilino no pagó nada”.

### Cómo lo usamos acá

Dos respuestas son **equivalentes** solo si hay entailment en **ambas** direcciones:

- pregunta + respuesta A  ⇒  respuesta B = entailment  
- pregunta + respuesta B  ⇒  respuesta A = entailment  

Si no, no se agrupan como el mismo significado.

In [3]:
RUTA_NLI = Path(
    "/home/data/legal/smulever/modelos/nli/"
    "mDeBERTa-v3-base-xnli-multilingual-nli-2mil7"
)

tok_nli = AutoTokenizer.from_pretrained(RUTA_NLI)
nli = AutoModelForSequenceClassification.from_pretrained(
    RUTA_NLI,
    torch_dtype=torch.bfloat16,
    device_map={"": 0},
)
nli.eval()

torch.cuda.synchronize()
print("nli:", next(nli.parameters()).device)
print("sync OK con los dos")
print(f"alloc MB: {torch.cuda.memory_allocated(0)/1024**2:.1f}")
print("id2label:", nli.config.id2label)

Loading weights: 100%|██████████| 202/202 [00:00<00:00, 2866.11it/s]

nli: cuda:0
sync OK con los dos
alloc MB: 15849.5
id2label: {0: 'entailment', 1: 'neutral', 2: 'contradiction'}


## Paso 1 — Muestreo de respuestas

Dada una pregunta $x$, generamos $M$ respuestas del modelo con sampling estocástico:

- `temperature = 1.0`
- `top_p = 1.0`
- `do_sample = True`

Objetivo: obtener variación léxica (y a veces semántica) entre respuestas a la misma pregunta. Esas $M$ secuencias son la entrada del clustering semántico.
-Dada una pregunta determinada, hacemos que el modelo genere 10 respuestas diferentes con su temperatura en 1.0

In [8]:
PREGUNTA = (
    "¿Puede un individuo vulnerar su derecho de intimidad "
    "al publicar datos privados de un tercero?"
)
M = 10
TEMPERATURE = 1.0
TOP_P = 1.0
MAX_NEW_TOKENS = 48

mensajes = [
    {
        "role": "system",
        "content": (
            "Respondé en UNA sola oración corta. "
            "Sin citar leyes, códigos ni tipificar delitos. "
            "Solo sí/no y el motivo mínimo."
        ),
    },
    {"role": "user", "content": PREGUNTA},
]

prompt = tok.apply_chat_template(
    mensajes,
    tokenize=False,
    add_generation_prompt=True,
)
inputs = tok(prompt, return_tensors="pt").to(model.device)

respuestas = []
with torch.inference_mode():
    for i in range(M):
        out = model.generate(
            **inputs,
            do_sample=True,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            max_new_tokens=MAX_NEW_TOKENS,
            pad_token_id=tok.pad_token_id,
        )
        gen = out[0, inputs["input_ids"].shape[-1] :]
        texto = tok.decode(gen, skip_special_tokens=True).strip()
        respuestas.append(texto)
        print(f"[{i+1:02d}] {texto}\n")

print(f"únicas: {len(set(respuestas))}/{M}")

[01] Sí, puede vulnerar el derecho de intimidad de un tercero al publicar datos privados sin su consentimiento.

[02] Sí, al publicar datos privados de un tercero sin su consentimiento, se puede considerar una vulneración de su derecho a la privacidad.

[03] Sí, al afectar a terceros.

[04] Sí, al violar la confianza y comprometer los datos personales de alguien sin consentimiento, el individuo puede vulnerar su derecho a la intimidad.

[05] Sí, al afectar el derecho a la imagen y honor del tercero.

[06] Sí, puede vulnerar el derecho a la intimidad de un tercero al publicar datos privados sobre ellos sin su consentimiento.

[07] Sí, al publicar datos privados de un tercero sin su consentimiento.

[08] Sí, puede vulnerar el derecho a la intimidad de un tercero.

[09] Sí, al difundir información personal sin consentimiento.

[10] Sí, al publicar datos personales de un tercero sin su consentimiento.

únicas: 10/10


## Paso 2 — Equivalencia semántica (entailment bidireccional)

Para cada par de respuestas distintas $(a, b)$:

1. $a \rightarrow b$: ¿la hipótesis $b$ se deduce de la premisa "pregunta + $a$"?
2. $b \rightarrow a$: ¿la hipótesis $a$ se deduce de la premisa "pregunta + $b$"?

- Si **ambas** dan entailment → mismo significado → mismo cluster.
- Si una es entailment y la otra neutral/contradiction → no son equivalentes.

In [10]:
import itertools
import torch.nn.functional as F

def nli_detalle(premisa: str, hipotesis: str) -> dict:
    enc = tok_nli(
        premisa,
        hipotesis,
        return_tensors="pt",
        truncation=True,
        max_length=512,
    )
    enc = {k: v.to(nli.device) for k, v in enc.items()}
    with torch.inference_mode():
        logits = nli(**enc).logits[0]
        probs = F.softmax(logits.float(), dim=-1)
    pred = int(probs.argmax().item())
    return {
        "label": nli.config.id2label[pred],
        "p_entailment": float(probs[0]),
        "p_neutral": float(probs[1]),
        "p_contradiction": float(probs[2]),
    }


comparaciones = []

for i, j in itertools.combinations(range(len(respuestas)), 2):
    a, b = respuestas[i], respuestas[j]
    print("=" * 80)
    print(f"PAR [{i+1:02d}] vs [{j+1:02d}]")
    print("-" * 80)

    # A → B
    prem_ab = f"{PREGUNTA} {a}"
    hip_ab = b
    det_ab = nli_detalle(prem_ab, hip_ab)
    print("Dirección A → B")
    print(f"  Premisa (pregunta + resp {i+1}): {prem_ab}")
    print(f"  Hipótesis (resp {j+1}):          {hip_ab}")
    print(f"  NLI: {det_ab['label']}")
    print(
        f"  probs: entail={det_ab['p_entailment']:.3f}  "
        f"neutral={det_ab['p_neutral']:.3f}  "
        f"contra={det_ab['p_contradiction']:.3f}"
    )
    print()

    # B → A
    prem_ba = f"{PREGUNTA} {b}"
    hip_ba = a
    det_ba = nli_detalle(prem_ba, hip_ba)
    print("Dirección B → A")
    print(f"  Premisa (pregunta + resp {j+1}): {prem_ba}")
    print(f"  Hipótesis (resp {i+1}):          {hip_ba}")
    print(f"  NLI: {det_ba['label']}")
    print(
        f"  probs: entail={det_ba['p_entailment']:.3f}  "
        f"neutral={det_ba['p_neutral']:.3f}  "
        f"contra={det_ba['p_contradiction']:.3f}"
    )
    print()

    eq = (det_ab["label"] == "entailment") and (det_ba["label"] == "entailment")
    print(f"  Equivalentes semánticos: {'SÍ' if eq else 'NO'}")

    comparaciones.append(
        {
            "i": i,
            "j": j,
            "label_ab": det_ab["label"],
            "label_ba": det_ba["label"],
            "p_ent_ab": det_ab["p_entailment"],
            "p_neu_ab": det_ab["p_neutral"],
            "p_con_ab": det_ab["p_contradiction"],
            "p_ent_ba": det_ba["p_entailment"],
            "p_neu_ba": det_ba["p_neutral"],
            "p_con_ba": det_ba["p_contradiction"],
            "equivalentes": eq,
        }
    )

print("=" * 80)
print(f"Total pares: {len(comparaciones)} | Total juicios NLI: {2 * len(comparaciones)}")

PAR [01] vs [02]
--------------------------------------------------------------------------------
Dirección A → B
  Premisa (pregunta + resp 1): ¿Puede un individuo vulnerar su derecho de intimidad al publicar datos privados de un tercero? Sí, puede vulnerar el derecho de intimidad de un tercero al publicar datos privados sin su consentimiento.
  Hipótesis (resp 2):          Sí, al publicar datos privados de un tercero sin su consentimiento, se puede considerar una vulneración de su derecho a la privacidad.
  NLI: entailment
  probs: entail=0.986  neutral=0.012  contra=0.001

Dirección B → A
  Premisa (pregunta + resp 2): ¿Puede un individuo vulnerar su derecho de intimidad al publicar datos privados de un tercero? Sí, al publicar datos privados de un tercero sin su consentimiento, se puede considerar una vulneración de su derecho a la privacidad.
  Hipótesis (resp 1):          Sí, puede vulnerar el derecho de intimidad de un tercero al publicar datos privados sin su consentimiento.
  

## Paso 2 (segunda mitad) — Clustering semántico (Algoritmo 1 del paper)

Agrupamos respuestas según equivalencia con el **representante** de cada cluster.

Reglas:

1. Recorremos las respuestas **en orden**.
2. El representante de un cluster es su **primer** elemento.
3. Una respuesta nueva entra a un cluster solo si es equivalente (entailment ida y vuelta) al **representante** de ese cluster.
4. Si no coincide con ningún representante, abre un **cluster nuevo** y pasa a ser su representante.

No usamos cierre transitivo entre miembros cualquiera del grupo: solo comparación contra el representante.

In [12]:
# Algoritmo 1 del paper: comparar solo contra el representante (primer elemento)

clusters = []  # lista de listas; clusters[k][0] = representante


def equivalentes_ij(i, j):
    """Reusa el resultado ya calculado en comparaciones."""
    a, b = min(i, j), max(i, j)
    for c in comparaciones:
        if c["i"] == a and c["j"] == b:
            return c["equivalentes"]
    return False


for i in range(len(respuestas)):
    assigned = False
    for cluster in clusters:
        rep = cluster[0]  # primer elemento = representante
        if equivalentes_ij(i, rep):
            cluster.append(i)
            assigned = True
            break
    if not assigned:
        clusters.append([i])

print(f"Cantidad de clusters: {len(clusters)}\n")
for k, idxs in enumerate(clusters, start=1):
    print(f"Cluster {k} (n={len(idxs)}): {[i+1 for i in idxs]}")
    print(f"  representante: [{idxs[0]+1:02d}] {respuestas[idxs[0]]}")
    for i in idxs:
        print(f"  [{i+1:02d}] {respuestas[i]}")
    print()

Cantidad de clusters: 4

Cluster 1 (n=7): [1, 2, 3, 4, 6, 9, 10]
  representante: [01] Sí, puede vulnerar el derecho de intimidad de un tercero al publicar datos privados sin su consentimiento.
  [01] Sí, puede vulnerar el derecho de intimidad de un tercero al publicar datos privados sin su consentimiento.
  [02] Sí, al publicar datos privados de un tercero sin su consentimiento, se puede considerar una vulneración de su derecho a la privacidad.
  [03] Sí, al afectar a terceros.
  [04] Sí, al violar la confianza y comprometer los datos personales de alguien sin consentimiento, el individuo puede vulnerar su derecho a la intimidad.
  [06] Sí, puede vulnerar el derecho a la intimidad de un tercero al publicar datos privados sobre ellos sin su consentimiento.
  [09] Sí, al difundir información personal sin consentimiento.
  [10] Sí, al publicar datos personales de un tercero sin su consentimiento.

Cluster 2 (n=1): [5]
  representante: [05] Sí, al afectar el derecho a la imagen y honor de

## Paso 3a — Log-probabilidad de cada respuesta

No regeneramos texto. Volvemos a pasar por Llama cada respuesta ya generada para obtener su log-probabilidad bajo el modelo (mismo prompt).

### Tokens y probabilidad

- Cada respuesta es una secuencia de **tokens**.
- En cada posición, el modelo asigna una **probabilidad** a ese token (número entre 0 y 1), dado el prompt y los tokens anteriores.

Ejemplo: si para el token "París" el modelo pone 40% de chances:

- Probabilidad = 0.40
- Log-prob = log(0.40) ≈ −0.92

El **log-prob** es el logaritmo de esa probabilidad (por eso suele ser negativo). No es otra cosa: es la misma chance, escrita en logaritmo.

### Por qué usamos log-probs

Una respuesta tiene muchos tokens. Si multiplicamos probabilidades chicas (0.4 × 0.3 × 0.2 × …) el número se va a cero.

En cambio **sumamos log-probs**:

- log-prob de la respuesta = logprob(token1) + logprob(token2) + …
- Después (paso 3b): probabilidad de la respuesta = exp(esa suma)

### Qué calculamos en esta celda

Para cada una de las M respuestas: la **suma** de los log-probs de sus tokens. Ese número (en general negativo) es la log-probabilidad de toda la secuencia.

In [13]:
import torch.nn.functional as F

def logprob_respuesta(prompt_text: str, respuesta: str) -> float:
    """Suma de log-probs de los tokens de `respuesta` dados `prompt_text`."""
    prompt_ids = tok(prompt_text, add_special_tokens=False, return_tensors="pt")["input_ids"][0]
    resp_ids = tok(respuesta, add_special_tokens=False, return_tensors="pt")["input_ids"][0]

    # secuencia completa: prompt + respuesta
    input_ids = torch.cat([prompt_ids, resp_ids]).unsqueeze(0).to(model.device)
    prompt_len = prompt_ids.shape[0]

    with torch.inference_mode():
        logits = model(input_ids).logits  # [1, seq, vocab]
        # el logit en posición t predice el token t+1
        log_probs = F.log_softmax(logits[0, prompt_len - 1 : -1], dim=-1)
        token_logprobs = log_probs.gather(
            1, resp_ids.to(model.device).unsqueeze(1)
        ).squeeze(1)

    return float(token_logprobs.sum().item())


# mismo prompt que usaste al generar
prompt = tok.apply_chat_template(
    mensajes,
    tokenize=False,
    add_generation_prompt=True,
)

logprobs = []
for i, r in enumerate(respuestas):
    lp = logprob_respuesta(prompt, r)
    logprobs.append(lp)
    print(f"[{i+1:02d}] logprob={lp:.4f}")
    print(f"     {r}\n")

[01] logprob=-8.0625
     Sí, puede vulnerar el derecho de intimidad de un tercero al publicar datos privados sin su consentimiento.

[02] logprob=-9.8125
     Sí, al publicar datos privados de un tercero sin su consentimiento, se puede considerar una vulneración de su derecho a la privacidad.

[03] logprob=-7.1562
     Sí, al afectar a terceros.

[04] logprob=-30.2500
     Sí, al violar la confianza y comprometer los datos personales de alguien sin consentimiento, el individuo puede vulnerar su derecho a la intimidad.

[05] logprob=-13.8750
     Sí, al afectar el derecho a la imagen y honor del tercero.

[06] logprob=-15.8125
     Sí, puede vulnerar el derecho a la intimidad de un tercero al publicar datos privados sobre ellos sin su consentimiento.

[07] logprob=-2.8594
     Sí, al publicar datos privados de un tercero sin su consentimiento.

[08] logprob=-8.2500
     Sí, puede vulnerar el derecho a la intimidad de un tercero.

[09] logprob=-8.3125
     Sí, al difundir información pe

-desglosemos cada respuesta en tokens

-vemos el logprobs de cada token

-hacemos la sumatoria de esos logprobs

In [19]:
import torch.nn.functional as F

def logprobs_por_token(prompt_text: str, respuesta: str):
    """Devuelve (tokens_str, logprobs_tensor, suma)."""
    prompt_ids = tok(prompt_text, add_special_tokens=False, return_tensors="pt")["input_ids"][0]
    resp_ids = tok(respuesta, add_special_tokens=False, return_tensors="pt")["input_ids"][0]

    input_ids = torch.cat([prompt_ids, resp_ids]).unsqueeze(0).to(model.device)
    prompt_len = prompt_ids.shape[0]

    with torch.inference_mode():
        logits = model(input_ids).logits
        log_probs = F.log_softmax(logits[0, prompt_len - 1 : -1], dim=-1)
        token_logprobs = log_probs.gather(
            1, resp_ids.to(model.device).unsqueeze(1)
        ).squeeze(1)

    tokens = [tok.decode([tid], skip_special_tokens=False) for tid in resp_ids.tolist()]
    return tokens, token_logprobs.detach().cpu(), float(token_logprobs.sum().item())


prompt = tok.apply_chat_template(
    mensajes,
    tokenize=False,
    add_generation_prompt=True,
)

print(f"Pregunta: {PREGUNTA}\n")

logprobs = []
for i, r in enumerate(respuestas):
    tokens, tlp, suma = logprobs_por_token(prompt, r)
    logprobs.append(suma)

    print("=" * 80)
    print(f"[{i+1:02d}] Respuesta: {r}")
    print(f"     n_tokens = {len(tokens)}")
    print("-" * 80)
    print(f"{'#':>4}  {'token':<20}  {'logprob':>10}")
    for t, (tok_str, lp) in enumerate(zip(tokens, tlp.tolist()), start=1):
        # repr para ver espacios/saltos
        print(f"{t:>4}  {repr(tok_str):<20}  {lp:>10.4f}")
    print("-" * 80)
    print(f"     SUMA logprobs = {suma:.4f}")
    print()

Pregunta: ¿Puede un individuo vulnerar su derecho de intimidad al publicar datos privados de un tercero?

[01] Respuesta: Sí, puede vulnerar el derecho de intimidad de un tercero al publicar datos privados sin su consentimiento.
     n_tokens = 27
--------------------------------------------------------------------------------
   #  token                    logprob
   1  'S'                      -0.0261
   2  'í'                      -0.0000
   3  ','                      -0.0011
   4  ' puede'                 -3.5156
   5  ' vulner'                -0.2090
   6  'ar'                     -0.0291
   7  ' el'                    -0.5234
   8  ' derecho'               -0.0005
   9  ' de'                    -0.1709
  10  ' intim'                 -0.0107
  11  'idad'                   -0.0000
  12  ' de'                    -0.4746
  13  ' un'                    -0.3926
  14  ' ter'                   -0.0033
  15  'c'                      -0.0000
  16  'ero'                    -0.0000
  17  ' 

## Paso 3b — De log-prob a probabilidad de secuencia

A cada log-prob de respuesta le aplicamos la exponencial:

- `p = exp(logprob)`
- `p` queda entre 0 y 1
- Es la probabilidad que el modelo asigna a **toda** esa respuesta (producto de las probs de sus tokens)

Ejemplo: si la log-prob de una respuesta es −1.0,

- `p = exp(−1.0) ≈ 0.37`

Respuestas más largas o menos “naturales” para el modelo suelen tener log-prob más negativa y por lo tanto `p` más chica.

In [14]:
import numpy as np

print(f"Pregunta: {PREGUNTA}\n")
print("=" * 80)

probs = []
for i, (respuesta, lp) in enumerate(zip(respuestas, logprobs)):
    p = float(np.exp(lp))
    probs.append(p)
    print(f"[{i+1:02d}] {respuesta}")
    print(f"     suma logprobs = {lp:.4f}  -->  exp(logprob) = {p:.6e}")
    print()

Pregunta: ¿Puede un individuo vulnerar su derecho de intimidad al publicar datos privados de un tercero?

[01] Sí, puede vulnerar el derecho de intimidad de un tercero al publicar datos privados sin su consentimiento.
     suma logprobs = -8.0625  -->  exp(logprob) = 3.151380e-04

[02] Sí, al publicar datos privados de un tercero sin su consentimiento, se puede considerar una vulneración de su derecho a la privacidad.
     suma logprobs = -9.8125  -->  exp(logprob) = 5.476277e-05

[03] Sí, al afectar a terceros.
     suma logprobs = -7.1562  -->  exp(logprob) = 7.799740e-04

[04] Sí, al violar la confianza y comprometer los datos personales de alguien sin consentimiento, el individuo puede vulnerar su derecho a la intimidad.
     suma logprobs = -30.2500  -->  exp(logprob) = 7.287724e-14

[05] Sí, al afectar el derecho a la imagen y honor del tercero.
     suma logprobs = -13.8750  -->  exp(logprob) = 9.422455e-07

[06] Sí, puede vulnerar el derecho a la intimidad de un tercero al publ

resultados expresados en notación científica

resultados expresado en decimales:

In [15]:
for i, (respuesta, lp, p) in enumerate(zip(respuestas, logprobs, probs)):
    print(f"[{i+1:02d}] {respuesta}")
    print(f"     suma logprobs = {lp:.4f}  -->  p = {p:.10f}")
    print()

[01] Sí, puede vulnerar el derecho de intimidad de un tercero al publicar datos privados sin su consentimiento.
     suma logprobs = -8.0625  -->  p = 0.0003151380

[02] Sí, al publicar datos privados de un tercero sin su consentimiento, se puede considerar una vulneración de su derecho a la privacidad.
     suma logprobs = -9.8125  -->  p = 0.0000547628

[03] Sí, al afectar a terceros.
     suma logprobs = -7.1562  -->  p = 0.0007799740

[04] Sí, al violar la confianza y comprometer los datos personales de alguien sin consentimiento, el individuo puede vulnerar su derecho a la intimidad.
     suma logprobs = -30.2500  -->  p = 0.0000000000

[05] Sí, al afectar el derecho a la imagen y honor del tercero.
     suma logprobs = -13.8750  -->  p = 0.0000009422

[06] Sí, puede vulnerar el derecho a la intimidad de un tercero al publicar datos privados sobre ellos sin su consentimiento.
     suma logprobs = -15.8125  -->  p = 0.0000001357

[07] Sí, al publicar datos privados de un tercero si

## Paso 3c — Probabilidad de cada cluster

Sumamos las probabilidades `p` de las respuestas que pertenecen a cada cluster. Esa suma es la masa del grupo (significado).

In [16]:
masas_cluster = []

print(f"Pregunta: {PREGUNTA}\n")
for k, idxs in enumerate(clusters, start=1):
    masa = sum(probs[i] for i in idxs)
    masas_cluster.append(masa)
    print(f"Cluster {k} (n={len(idxs)}): índices {[i+1 for i in idxs]}")
    for i in idxs:
        print(f"  [{i+1:02d}] p={probs[i]:.10f}  |  {respuestas[i]}")
    print(f"  --> masa del cluster = {masa:.10f}\n")

Pregunta: ¿Puede un individuo vulnerar su derecho de intimidad al publicar datos privados de un tercero?

Cluster 1 (n=7): índices [1, 2, 3, 4, 6, 9, 10]
  [01] p=0.0003151380  |  Sí, puede vulnerar el derecho de intimidad de un tercero al publicar datos privados sin su consentimiento.
  [02] p=0.0000547628  |  Sí, al publicar datos privados de un tercero sin su consentimiento, se puede considerar una vulneración de su derecho a la privacidad.
  [03] p=0.0007799740  |  Sí, al afectar a terceros.
  [04] p=0.0000000000  |  Sí, al violar la confianza y comprometer los datos personales de alguien sin consentimiento, el individuo puede vulnerar su derecho a la intimidad.
  [06] p=0.0000001357  |  Sí, puede vulnerar el derecho a la intimidad de un tercero al publicar datos privados sobre ellos sin su consentimiento.
  [09] p=0.0002454297  |  Sí, al difundir información personal sin consentimiento.
  [10] p=0.0014571832  |  Sí, al publicar datos personales de un tercero sin su consentimiento.

## Paso 3d — Normalizar probabilidades de cluster

Dividimos la probabilidad de cada cluster por la suma de todas las probabilidades. Así obtenemos una distribución sobre significados (suma 1).

In [17]:
total = sum(masas_cluster)
probs_cluster = [m / total for m in masas_cluster]

print(f"Suma de masas (antes): {total:.10f}\n")
for k, (masa, pc) in enumerate(zip(masas_cluster, probs_cluster), start=1):
    print(
        f"Cluster {k}: masa={masa:.10f}  -->  "
        f"p_normalizada={pc:.6f}  ({100*pc:.2f}%)"
    )
print(f"\nSuma normalizada: {sum(probs_cluster):.6f}")

Suma de masas (antes): 0.0604193886

Cluster 1: masa=0.0028526234  -->  p_normalizada=0.047214  (4.72%)
Cluster 2: masa=0.0000009422  -->  p_normalizada=0.000016  (0.00%)
Cluster 3: masa=0.0573045644  -->  p_normalizada=0.948447  (94.84%)
Cluster 4: masa=0.0002612586  -->  p_normalizada=0.004324  (0.43%)

Suma normalizada: 1.000000


## Paso 3e — Semantic entropy

$$
SE = -\sum_{c=1}^{k} p_c \ln(p_c)
$$

donde $p_c$ es la probabilidad normalizada del cluster $c$ y $k$ es la cantidad de clusters. Más alto = más incertidumbre semántica.

In [18]:
import math

SE = 0.0
for pc in probs_cluster:
    if pc > 0:
        SE += pc * math.log(pc)  # ln natural
SE = -SE

print(f"k (clusters): {len(probs_cluster)}")
for k, pc in enumerate(probs_cluster, start=1):
    term = -pc * math.log(pc) if pc > 0 else 0.0
    print(f"  cluster {k}: p={pc:.6f}  |  -p*ln(p)={term:.6f}")
print(f"\nSemantic entropy SE = {SE:.6f}")

k (clusters): 4
  cluster 1: p=0.047214  |  -p*ln(p)=0.144147
  cluster 2: p=0.000016  |  -p*ln(p)=0.000173
  cluster 3: p=0.948447  |  -p*ln(p)=0.050201
  cluster 4: p=0.004324  |  -p*ln(p)=0.023538

Semantic entropy SE = 0.218059


## Interpretación del resultado

El pipeline cerró bien: muestreo → NLI → clusters → probs → semantic entropy.

### De respuestas a significados

Hasta el clustering trabajamos con **respuestas aisladas** (secuencias de tokens distintas).

Desde el clustering en adelante, cada cluster representa un **significado**: distintas formas de decir lo mismo quedan juntas.

Por eso la semantic entropy no mide “cuánto varía el texto”, sino **cómo se reparte la probabilidad entre significados distintos**.

### Qué significan los porcentajes de cada cluster

Después de normalizar, cada cluster tiene una probabilidad (que suma 1 entre todos).

Esa probabilidad **no** es “qué tan correcta es la respuesta en el mundo real”.

Es la fracción de probabilidad que el método asigna a **ese significado**, estimada a partir de las respuestas muestreadas:

1. Cada respuesta tiene una probabilidad bajo el modelo (qué tan probable es esa secuencia de tokens).
2. Se suman las del mismo cluster → probabilidad del **significado**.
3. Se normaliza → cómo se reparte esa masa entre los significados observados.

En nuestro caso:

- Cluster 3 ≈ 95%: casi toda la masa en ese significado.
- Cluster 1 ≈ 5%: el grupo con más paráfrasis, pero poca probabilidad.
- Clusters 2 y 4 ≈ 0%: significados muy poco probables bajo el modelo.

### Por qué el SE salió bajo

La semantic entropy mide qué tan repartida está la probabilidad entre significados.

- Si un significado concentra casi todo (como el cluster 3 con ~95%), el SE es **bajo**.
- Si varios significados se reparte la masa, el SE es **alto**.

Acá SE ≈ 0.22 porque el significado del cluster 3 domina.

### Matiz importante (sesgo por longitud)

El cluster 1 agrupa **más respuestas** del mismo significado (mayoría por conteo).  
El cluster 3 tiene **una** respuesta, más corta, pero mucha más probabilidad conjunta.

Eso pasa porque usamos `exp(suma de logprobs)`: las secuencias cortas suelen ser más probables que las largas, aunque digan casi lo mismo.

Entonces:

- El método está bien aplicado.
- El SE bajo refleja una distribución muy concentrada en un significado (el del cluster 3).
- Parte de esa concentración viene del **sesgo por longitud**, no solo de que “todo el mundo dijo lo mismo” en el cluster más poblado.

El paper también discute un baseline con logprobs normalizados por cantidad de tokens (length-normalised) para mirar este efecto. Dividir esa suma por la cantidad de tokens → promedio de logprob por token.

## Ablación — SE con logprobs length-normalised

Misma clusterización (significados iguales).  
Cambia el peso de cada respuesta: en lugar de `exp(suma de logprobs)`, usamos `exp(suma / n_tokens)` (promedio por token).  
Después: sumar por cluster → normalizar → semantic entropy. Comparar con el SE original.

In [22]:
import torch.nn.functional as F

def desglose_respuesta(prompt_text: str, respuesta: str):
    prompt_ids = tok(prompt_text, add_special_tokens=False, return_tensors="pt")["input_ids"][0]
    resp_ids = tok(respuesta, add_special_tokens=False, return_tensors="pt")["input_ids"][0]
    input_ids = torch.cat([prompt_ids, resp_ids]).unsqueeze(0).to(model.device)
    prompt_len = prompt_ids.shape[0]

    with torch.inference_mode():
        logits = model(input_ids).logits
        log_probs = F.log_softmax(logits[0, prompt_len - 1 : -1], dim=-1)
        token_logprobs = log_probs.gather(
            1, resp_ids.to(model.device).unsqueeze(1)
        ).squeeze(1)

    tokens = [tok.decode([tid], skip_special_tokens=False) for tid in resp_ids.tolist()]
    tlp = token_logprobs.detach().cpu()
    suma = float(tlp.sum().item())
    n_tok = len(tokens)
    media = suma / n_tok
    return tokens, tlp, suma, media, n_tok


prompt = tok.apply_chat_template(
    mensajes,
    tokenize=False,
    add_generation_prompt=True,
)

print(f"Pregunta: {PREGUNTA}\n")

sumas = []
medias = []
n_tokens_list = []

for i, r in enumerate(respuestas):
    tokens, tlp, suma, media, n_tok = desglose_respuesta(prompt, r)
    sumas.append(suma)
    medias.append(media)
    n_tokens_list.append(n_tok)

    print("=" * 80)
    print(f"[{i+1:02d}] {r}")
    print(f"     n_tokens = {n_tok}")
    print("-" * 80)
    print(f"{'#':>4}  {'token':<20}  {'logprob':>10}")
    for t, (tok_str, lp) in enumerate(zip(tokens, tlp.tolist()), start=1):
        print(f"{t:>4}  {repr(tok_str):<20}  {lp:>10.4f}")
    print("-" * 80)
    print(f"     SUMA  = {suma:.4f}")
    print(f"     MEDIA = {media:.4f}  (= suma / {n_tok})")
    print()

Pregunta: ¿Puede un individuo vulnerar su derecho de intimidad al publicar datos privados de un tercero?

[01] Sí, puede vulnerar el derecho de intimidad de un tercero al publicar datos privados sin su consentimiento.
     n_tokens = 27
--------------------------------------------------------------------------------
   #  token                    logprob
   1  'S'                      -0.0261
   2  'í'                      -0.0000
   3  ','                      -0.0011
   4  ' puede'                 -3.5156
   5  ' vulner'                -0.2090
   6  'ar'                     -0.0291
   7  ' el'                    -0.5234
   8  ' derecho'               -0.0005
   9  ' de'                    -0.1709
  10  ' intim'                 -0.0107
  11  'idad'                   -0.0000
  12  ' de'                    -0.4746
  13  ' un'                    -0.3926
  14  ' ter'                   -0.0033
  15  'c'                      -0.0000
  16  'ero'                    -0.0000
  17  ' al'        

## Ablación B — De media a peso length-normalised

`p_len = exp(media)`. Ese peso ya no es la probabilidad conjunta de la secuencia; es `exp` del logprob promedio por token.

In [23]:
import math

probs_len = []
print(f"Pregunta: {PREGUNTA}\n")
for i, (r, suma, media, n_tok) in enumerate(zip(respuestas, sumas, medias, n_tokens_list)):
    p_joint = math.exp(suma)
    p_len = math.exp(media)
    probs_len.append(p_len)
    print(f"[{i+1:02d}] {r}")
    print(f"     n_tok={n_tok}  suma={suma:.4f}  media={media:.4f}")
    print(f"     p_joint=exp(suma)={p_joint:.6e}  |  p_len=exp(media)={p_len:.6e}\n")

Pregunta: ¿Puede un individuo vulnerar su derecho de intimidad al publicar datos privados de un tercero?

[01] Sí, puede vulnerar el derecho de intimidad de un tercero al publicar datos privados sin su consentimiento.
     n_tok=27  suma=-8.0625  media=-0.2986
     p_joint=exp(suma)=3.151380e-04  |  p_len=exp(media)=7.418478e-01

[02] Sí, al publicar datos privados de un tercero sin su consentimiento, se puede considerar una vulneración de su derecho a la privacidad.
     n_tok=34  suma=-9.8125  media=-0.2886
     p_joint=exp(suma)=5.476277e-05  |  p_len=exp(media)=7.493097e-01

[03] Sí, al afectar a terceros.
     n_tok=10  suma=-7.1562  media=-0.7156
     p_joint=exp(suma)=7.799740e-04  |  p_len=exp(media)=4.888865e-01

[04] Sí, al violar la confianza y comprometer los datos personales de alguien sin consentimiento, el individuo puede vulnerar su derecho a la intimidad.
     n_tok=36  suma=-30.2500  media=-0.8403
     p_joint=exp(suma)=7.287724e-14  |  p_len=exp(media)=4.315906e-01



## Ablación C — Masa por cluster (pesos length-normalised)

Sumamos `p_len` dentro de cada cluster (mismos clusters del Algoritmo 1).

In [24]:
masas_len = []
for k, idxs in enumerate(clusters, start=1):
    masa = sum(probs_len[i] for i in idxs)
    masas_len.append(masa)
    print(f"Cluster {k} (n={len(idxs)}): {[i+1 for i in idxs]}")
    for i in idxs:
        print(f"  [{i+1:02d}] p_len={probs_len[i]:.6e}  |  {respuestas[i]}")
    print(f"  --> masa_len = {masa:.6e}\n")

Cluster 1 (n=7): [1, 2, 3, 4, 6, 9, 10]
  [01] p_len=7.418478e-01  |  Sí, puede vulnerar el derecho de intimidad de un tercero al publicar datos privados sin su consentimiento.
  [02] p_len=7.493097e-01  |  Sí, al publicar datos privados de un tercero sin su consentimiento, se puede considerar una vulneración de su derecho a la privacidad.
  [03] p_len=4.888865e-01  |  Sí, al afectar a terceros.
  [04] p_len=4.315906e-01  |  Sí, al violar la confianza y comprometer los datos personales de alguien sin consentimiento, el individuo puede vulnerar su derecho a la intimidad.
  [06] p_len=5.903242e-01  |  Sí, puede vulnerar el derecho a la intimidad de un tercero al publicar datos privados sobre ellos sin su consentimiento.
  [09] p_len=5.275967e-01  |  Sí, al difundir información personal sin consentimiento.
  [10] p_len=7.091062e-01  |  Sí, al publicar datos personales de un tercero sin su consentimiento.
  --> masa_len = 4.238662e+00

Cluster 2 (n=1): [5]
  [05] p_len=4.626274e-01  |  Sí,

## Ablación D — Normalizar masas length-normalised

In [25]:
total_len = sum(masas_len)
probs_cluster_len = [m / total_len for m in masas_len]

print(f"{'Cluster':<10} {'p_SE_original':>14} {'p_SE_len_norm':>14}")
for k, (p0, p1) in enumerate(zip(probs_cluster, probs_cluster_len), start=1):
    print(f"{k:<10} {p0:14.6f} {p1:14.6f}")
print(f"\nSuma len-norm: {sum(probs_cluster_len):.6f}")

Cluster     p_SE_original  p_SE_len_norm
1                0.047214       0.684327
2                0.000016       0.074691
3                0.948447       0.138892
4                0.004324       0.102090

Suma len-norm: 1.000000


## Ablación E — Semantic entropy con pesos length-normalised

Misma fórmula de SE, pero con las probs normalizadas de la ablación. Comparar con el SE original.

In [27]:
import math

SE_len = -sum(p * math.log(p) for p in probs_cluster_len if p > 0)

print(f"k = {len(probs_cluster_len)}")
for k, pc in enumerate(probs_cluster_len, start=1):
    term = -pc * math.log(pc) if pc > 0 else 0.0
    print(f"  cluster {k}: p={pc:.6f}  |  -p*ln(p)={term:.6f}")

print(f"\nSE original:          {SE:.6f}")
print(f"SE length-normalised: {SE_len:.6f}")
print(f"Diferencia:           {SE_len - SE:.6f}")

k = 4
  cluster 1: p=0.684327  |  -p*ln(p)=0.259578
  cluster 2: p=0.074691  |  -p*ln(p)=0.193778
  cluster 3: p=0.138892  |  -p*ln(p)=0.274181
  cluster 4: p=0.102090  |  -p*ln(p)=0.232959

SE original:          0.218059
SE length-normalised: 0.960496
Diferencia:           0.742437


## Por qué subió tanto el SE con length-norm

### Recordatorio rápido

- **Cluster:** grupo de respuestas que el método trata como el mismo significado.
- **Algoritmo 1 (el del paper):** una respuesta nueva entra a un grupo solo si es equivalente (NLI ida y vuelta) al **primer** miembro de ese grupo (el representante). Si no, abre un grupo nuevo.
- **Length-norm:** en lugar de usar la probabilidad de toda la secuencia, usamos un peso basado en el logprob **promedio** por token. Eso evita que las respuestas cortas aplasten a las largas.

### Qué vimos

Con el método original (probabilidad conjunta) el SE era **bajo** (~0.22), porque una sola respuesta corta se llevaba casi toda la probabilidad.

Con length-norm el SE subió mucho (~0.96).

### La causa principal: demasiados grupos

El Algoritmo 1 armó **4 grupos**, aunque a ojo casi todas las respuestas dicen lo mismo (“sí, sin consentimiento…”).

Ejemplo del problema:

- El grupo grande empieza con la respuesta 01 (representante).
- Las respuestas 07 y 08 dicen casi lo mismo, pero el NLI **no** las marcó equivalentes a la 01.
- Como la regla solo mira al representante, 07 y 08 abrieron **grupos propios**.

O sea: no es que el modelo baraje 4 ideas distintas. Es que la regla de agrupación fue **estricta** y partió de más.

### Por qué eso dispara el SE con length-norm

Con length-norm, los pesos de las respuestas quedan más parecidos. Entonces importa cuántos grupos hay y cómo se reparte la masa entre ellos.

Con 4 grupos “rivales”, la probabilidad se reparte → el SE sube.

Si hubiéramos agrupado de forma menos estricta (por ejemplo uniendo también cuando hay una cadena: A equivalente a B y B equivalente a C), muchas de esas respuestas irían al **mismo** significado y el SE con length-norm bajaría.

### En una frase

El SE length-norm alto no refleja tanta incertidumbre real del modelo; refleja sobre todo que la clusterización estricta inventó más significados de los que hay, y length-norm les dio peso de verdad.

ACLARACIÓN: LO SIGUIENTE NO ESTÁ INCLUÍDO EN EL PAPER DE ESTUDIO

## Variante de clusterización — cierre transitivo

**No es el Algoritmo 1 del paper.** Es una alternativa para comparar.

### Algoritmo 1 (paper)
Una respuesta entra a un grupo solo si es equivalente (NLI ida y vuelta) al **representante** (primer miembro). Si no, abre un grupo nuevo.

### Esta variante (transitiva)
Usamos las mismas comparaciones NLI ya calculadas.

Regla: si A es equivalente a B, van al mismo grupo.  
Si además B es equivalente a C, **A, B y C** quedan juntos, aunque A y C no se hayan marcado equivalentes entre sí.

Objetivo: evitar partir de más respuestas que, por cadena, dicen lo mismo.

### Qué hacemos después
Con esos grupos nuevos, repetimos el cálculo de semantic entropy:

1. Con probabilidad conjunta (como el SE original).
2. Con pesos length-norm (como la ablación).

Así vemos si el SE baja al tener menos “significados” artificiales.

In [28]:
from collections import defaultdict

# Clusterización transitiva (variante; no es Algoritmo 1)
n = len(respuestas)
parent = list(range(n))


def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x


def union(x, y):
    rx, ry = find(x), find(y)
    if rx != ry:
        parent[ry] = rx


def equivalentes_ij(i, j):
    """Reusa el resultado ya calculado en comparaciones."""
    a, b = min(i, j), max(i, j)
    for c in comparaciones:
        if c["i"] == a and c["j"] == b:
            return c["equivalentes"]
    return False


for c in comparaciones:
    if c["equivalentes"]:
        union(c["i"], c["j"])

# armar grupos (ordenados por primer índice, como salida estable)
grupos = defaultdict(list)
for i in range(n):
    grupos[find(i)].append(i)

clusters_trans = [sorted(idxs) for idxs in grupos.values()]
clusters_trans.sort(key=lambda idxs: idxs[0])

print(f"Cantidad de clusters: {len(clusters_trans)}\n")
for k, idxs in enumerate(clusters_trans, start=1):
    print(f"Cluster {k} (n={len(idxs)}): {[i+1 for i in idxs]}")
    print(f"  representante: [{idxs[0]+1:02d}] {respuestas[idxs[0]]}")
    for i in idxs:
        print(f"  [{i+1:02d}] {respuestas[i]}")
    print()

Cantidad de clusters: 2

Cluster 1 (n=9): [1, 2, 3, 4, 6, 7, 8, 9, 10]
  representante: [01] Sí, puede vulnerar el derecho de intimidad de un tercero al publicar datos privados sin su consentimiento.
  [01] Sí, puede vulnerar el derecho de intimidad de un tercero al publicar datos privados sin su consentimiento.
  [02] Sí, al publicar datos privados de un tercero sin su consentimiento, se puede considerar una vulneración de su derecho a la privacidad.
  [03] Sí, al afectar a terceros.
  [04] Sí, al violar la confianza y comprometer los datos personales de alguien sin consentimiento, el individuo puede vulnerar su derecho a la intimidad.
  [06] Sí, puede vulnerar el derecho a la intimidad de un tercero al publicar datos privados sobre ellos sin su consentimiento.
  [07] Sí, al publicar datos privados de un tercero sin su consentimiento.
  [08] Sí, puede vulnerar el derecho a la intimidad de un tercero.
  [09] Sí, al difundir información personal sin consentimiento.
  [10] Sí, al publica

## Variante transitiva — Desglose por token

Para cada respuesta: tokens, logprob de cada uno, **suma** y **promedio** (suma / n_tokens).

In [30]:
import torch.nn.functional as F

def desglose_respuesta(prompt_text: str, respuesta: str):
    prompt_ids = tok(prompt_text, add_special_tokens=False, return_tensors="pt")["input_ids"][0]
    resp_ids = tok(respuesta, add_special_tokens=False, return_tensors="pt")["input_ids"][0]
    input_ids = torch.cat([prompt_ids, resp_ids]).unsqueeze(0).to(model.device)
    prompt_len = prompt_ids.shape[0]

    with torch.inference_mode():
        logits = model(input_ids).logits
        log_probs = F.log_softmax(logits[0, prompt_len - 1 : -1], dim=-1)
        token_logprobs = log_probs.gather(
            1, resp_ids.to(model.device).unsqueeze(1)
        ).squeeze(1)

    tokens = [tok.decode([tid], skip_special_tokens=False) for tid in resp_ids.tolist()]
    tlp = token_logprobs.detach().cpu()
    suma = float(tlp.sum().item())
    n_tok = len(tokens)
    media = suma / n_tok
    return tokens, tlp, suma, media, n_tok


prompt = tok.apply_chat_template(
    mensajes,
    tokenize=False,
    add_generation_prompt=True,
)

print(f"Pregunta: {PREGUNTA}\n")

sumas = []
medias = []
n_tokens_list = []

for i, r in enumerate(respuestas):
    tokens, tlp, suma, media, n_tok = desglose_respuesta(prompt, r)
    sumas.append(suma)
    medias.append(media)
    n_tokens_list.append(n_tok)

    print("=" * 80)
    print(f"[{i+1:02d}] {r}")
    print(f"     n_tokens = {n_tok}")
    print("-" * 80)
    print(f"{'#':>4}  {'token':<20}  {'logprob':>10}")
    for t, (tok_str, lp) in enumerate(zip(tokens, tlp.tolist()), start=1):
        print(f"{t:>4}  {repr(tok_str):<20}  {lp:>10.4f}")
    print("-" * 80)
    print(f"     SUMA     = {suma:.4f}")
    print(f"     PROMEDIO = {media:.4f}  (= suma / {n_tok})")
    print()

Pregunta: ¿Puede un individuo vulnerar su derecho de intimidad al publicar datos privados de un tercero?

[01] Sí, puede vulnerar el derecho de intimidad de un tercero al publicar datos privados sin su consentimiento.
     n_tokens = 27
--------------------------------------------------------------------------------
   #  token                    logprob
   1  'S'                      -0.0261
   2  'í'                      -0.0000
   3  ','                      -0.0011
   4  ' puede'                 -3.5156
   5  ' vulner'                -0.2090
   6  'ar'                     -0.0291
   7  ' el'                    -0.5234
   8  ' derecho'               -0.0005
   9  ' de'                    -0.1709
  10  ' intim'                 -0.0107
  11  'idad'                   -0.0000
  12  ' de'                    -0.4746
  13  ' un'                    -0.3926
  14  ' ter'                   -0.0033
  15  'c'                      -0.0000
  16  'ero'                    -0.0000
  17  ' al'        

## Variante transitiva — exp(suma) y exp(promedio)

- `p_joint = exp(suma)` → probabilidad conjunta (método del paper)
- `p_len = exp(promedio)` → peso length-norm

In [33]:
import math

probs = []      # exp(suma)
probs_len = []  # exp(promedio)

print(f"Pregunta: {PREGUNTA}\n")
for i, (r, suma, media, n_tok) in enumerate(
    zip(respuestas, sumas, medias, n_tokens_list)
):
    p_joint = math.exp(suma)
    p_len = math.exp(media)
    probs.append(p_joint)
    probs_len.append(p_len)

    print(f"[{i+1:02d}] {r}")
    print(f"     n_tokens = {n_tok}")
    print(f"     suma     = {suma:.4f}  -->  exp(suma)     = {p_joint:.10f}")
    print(f"     promedio = {media:.4f}  -->  exp(promedio) = {p_len:.10f}")
    print()

Pregunta: ¿Puede un individuo vulnerar su derecho de intimidad al publicar datos privados de un tercero?

[01] Sí, puede vulnerar el derecho de intimidad de un tercero al publicar datos privados sin su consentimiento.
     n_tokens = 27
     suma     = -8.0625  -->  exp(suma)     = 0.0003151380
     promedio = -0.2986  -->  exp(promedio) = 0.7418478497

[02] Sí, al publicar datos privados de un tercero sin su consentimiento, se puede considerar una vulneración de su derecho a la privacidad.
     n_tokens = 34
     suma     = -9.8125  -->  exp(suma)     = 0.0000547628
     promedio = -0.2886  -->  exp(promedio) = 0.7493096664

[03] Sí, al afectar a terceros.
     n_tokens = 10
     suma     = -7.1562  -->  exp(suma)     = 0.0007799740
     promedio = -0.7156  -->  exp(promedio) = 0.4888864623

[04] Sí, al violar la confianza y comprometer los datos personales de alguien sin consentimiento, el individuo puede vulnerar su derecho a la intimidad.
     n_tokens = 36
     suma     = -30.2500

## Variante transitiva — Masa por cluster

Sumamos `exp(suma)` y `exp(promedio)` dentro de cada cluster transitivo.

In [35]:
masas_trans = []
masas_trans_len = []

print(f"Pregunta: {PREGUNTA}\n")
for k, idxs in enumerate(clusters_trans, start=1):
    masa = sum(probs[i] for i in idxs)
    masa_len = sum(probs_len[i] for i in idxs)
    masas_trans.append(masa)
    masas_trans_len.append(masa_len)

    print(f"Cluster {k} (n={len(idxs)}): {[i+1 for i in idxs]}")
    for i in idxs:
        print(
            f"  [{i+1:02d}] exp(suma)={probs[i]:.10f}  |  "
            f"exp(promedio)={probs_len[i]:.10f}  |  {respuestas[i]}"
        )
    print(f"  --> masa exp(suma)     = {masa:.10f}")
    print(f"  --> masa exp(promedio) = {masa_len:.10f}\n")

Pregunta: ¿Puede un individuo vulnerar su derecho de intimidad al publicar datos privados de un tercero?

Cluster 1 (n=9): [1, 2, 3, 4, 6, 7, 8, 9, 10]
  [01] exp(suma)=0.0003151380  |  exp(promedio)=0.7418478497  |  Sí, puede vulnerar el derecho de intimidad de un tercero al publicar datos privados sin su consentimiento.
  [02] exp(suma)=0.0000547628  |  exp(promedio)=0.7493096664  |  Sí, al publicar datos privados de un tercero sin su consentimiento, se puede considerar una vulneración de su derecho a la privacidad.
  [03] exp(suma)=0.0007799740  |  exp(promedio)=0.4888864623  |  Sí, al afectar a terceros.
  [04] exp(suma)=0.0000000000  |  exp(promedio)=0.4315906205  |  Sí, al violar la confianza y comprometer los datos personales de alguien sin consentimiento, el individuo puede vulnerar su derecho a la intimidad.
  [06] exp(suma)=0.0000001357  |  exp(promedio)=0.5903242402  |  Sí, puede vulnerar el derecho a la intimidad de un tercero al publicar datos privados sobre ellos sin su c

## Variante transitiva — Normalizar (las dos versiones)

Dividimos cada masa por la suma de todas las masas de su versión:

1. **Prob conjunta:** masas de `exp(suma)` → probs normalizadas de cada significado.
2. **Length-norm:** masas de `exp(promedio)` → probs normalizadas de cada significado.

En ambos casos las probs de los clusters suman 1. Eso es la distribución sobre significados que entra a la semantic entropy.

In [36]:
total_trans = sum(masas_trans)
total_trans_len = sum(masas_trans_len)

probs_cluster_trans = [m / total_trans for m in masas_trans]
probs_cluster_trans_len = [m / total_trans_len for m in masas_trans_len]

print(f"Suma masas exp(suma):     {total_trans:.10f}")
print(f"Suma masas exp(promedio): {total_trans_len:.10f}\n")

print(f"{'Cluster':<10} {'p_conjunta':>14} {'p_len_norm':>14}")
for k, (p0, p1) in enumerate(zip(probs_cluster_trans, probs_cluster_trans_len), start=1):
    print(
        f"{k:<10} {p0:14.6f} ({100*p0:5.2f}%)  "
        f"{p1:14.6f} ({100*p1:5.2f}%)"
    )

print(f"\nSuma conjunta: {sum(probs_cluster_trans):.6f}")
print(f"Suma len-norm: {sum(probs_cluster_trans_len):.6f}")

Suma masas exp(suma):     0.0604193886
Suma masas exp(promedio): 6.1939091772

Cluster        p_conjunta     p_len_norm
1                0.999984 (100.00%)        0.925309 (92.53%)
2                0.000016 ( 0.00%)        0.074691 ( 7.47%)

Suma conjunta: 1.000000
Suma len-norm: 1.000000


## Variante transitiva — Semantic entropy (las dos versiones)

Misma fórmula para ambas:

SE = − Σ p_c × ln(p_c)

donde p_c son las probs normalizadas de cada cluster.

Calculamos:

1. **SE transitiva + conjunta** → con probs de `exp(suma)`
2. **SE transitiva + length-norm** → con probs de `exp(promedio)`

Y comparamos con los SE del Algoritmo 1 (si están en memoria).

In [37]:
import math

SE_trans = -sum(p * math.log(p) for p in probs_cluster_trans if p > 0)
SE_trans_len = -sum(p * math.log(p) for p in probs_cluster_trans_len if p > 0)

print("Desglose −p·ln(p) por cluster\n")
print(f"{'Cluster':<10} {'conjunta':>12} {'len_norm':>12}")
for k, (p0, p1) in enumerate(zip(probs_cluster_trans, probs_cluster_trans_len), start=1):
    t0 = -p0 * math.log(p0) if p0 > 0 else 0.0
    t1 = -p1 * math.log(p1) if p1 > 0 else 0.0
    print(f"{k:<10} {t0:12.6f} {t1:12.6f}")

print("\n" + "=" * 60)
print("Comparación final")
print("=" * 60)
print(f"SE transitiva + conjunta:       {SE_trans:.6f}")
print(f"SE transitiva + length-norm:    {SE_trans_len:.6f}")
if "SE" in dir():
    print(f"SE Algo 1 + conjunta:           {SE:.6f}")
if "SE_len" in dir():
    print(f"SE Algo 1 + length-norm:        {SE_len:.6f}")

Desglose −p·ln(p) por cluster

Cluster        conjunta     len_norm
1              0.000016     0.071829
2              0.000173     0.193778

Comparación final
SE transitiva + conjunta:       0.000188
SE transitiva + length-norm:    0.265607
SE Algo 1 + conjunta:           0.218059
SE Algo 1 + length-norm:        0.960496


In [45]:
from IPython.display import HTML, display

filas = [
    ("Algoritmo 1", "Prob. conjunta", 0.218059),
    ("Algoritmo 1", "Length-norm", 0.960496),
    ("Transitiva", "Prob. conjunta", 0.000188),
    ("Transitiva", "Length-norm", 0.265607),
]

html = """
<style>
.se-wrap {
  font-family: "Segoe UI", system-ui, sans-serif;
  max-width: 640px;
  margin: 1rem 0;
  border: 1px solid #d8d8d4;
  border-radius: 10px;
  overflow: hidden;
  background: #fafaf8;
}
.se-wrap h3 {
  margin: 0;
  padding: 14px 18px;
  background: #1f2a24;
  color: #f4f1ea;
  font-size: 15px;
  font-weight: 600;
  letter-spacing: 0.02em;
}
.se-wrap table {
  width: 100%;
  border-collapse: collapse;
  font-size: 14px;
}
.se-wrap th {
  text-align: left;
  padding: 10px 18px;
  background: #e8ebe6;
  color: #333;
  font-weight: 600;
  border-bottom: 1px solid #d0d4ce;
}
.se-wrap td {
  padding: 11px 18px;
  border-bottom: 1px solid #e6e6e2;
  color: #222;
}
.se-wrap tr:last-child td { border-bottom: none; }
.se-wrap .num {
  font-variant-numeric: tabular-nums;
  font-family: ui-monospace, Menlo, Consolas, monospace;
  text-align: right;
}
.se-wrap .low {
  background: #e7f2ea;
  font-weight: 600;
}
.se-wrap .high {
  background: #f7ebe6;
}
.se-wrap .note {
  padding: 10px 18px 14px;
  font-size: 12px;
  color: #666;
  background: #f3f3ef;
  border-top: 1px solid #e0e0da;
}
</style>
<div class="se-wrap">
  <h3>Comparación final — Semantic Entropy</h3>
  <table>
    <thead>
      <tr>
        <th>Clusterización</th>
        <th>Pesos</th>
        <th style="text-align:right">SE</th>
      </tr>
    </thead>
    <tbody>
"""

vals = [r[2] for r in filas]
vmin, vmax = min(vals), max(vals)

for cluster, pesos, se in filas:
    cls = "low" if se == vmin else ("high" if se == vmax else "")
    html += f"""
      <tr class="{cls}">
        <td>{cluster}</td>
        <td>{pesos}</td>
        <td class="num">{se:.6f}</td>
      </tr>
    """

html += """
    </tbody>
  </table>
</div>
"""

display(HTML(html))

Clusterización,Pesos,SE
Algoritmo 1,Prob. conjunta,0.218059
Algoritmo 1,Length-norm,0.960496
Transitiva,Prob. conjunta,0.000188
Transitiva,Length-norm,0.265607
